# 09 — SAJU ML V4 Unified DEV Wave 2 Subject Roster Freeze

**Version:** `SAJU_ML_V4_UNIFIED_DEV_WAVE2_SUBJECT_ROSTER_FREEZE_20260816`

Purpose: freeze a completely fresh Wave 2 subject roster after Wave 1 produced 73 usable pairs.

Predeclared Wave 2 roster target, derived only from Wave 1 axis deficits / observed pairability:

- COMPETITIVE: 3
- PROJECT: 8
- STATUS: 33
- TOTAL: 44

Hard rules:
- Do not replace any Wave 1 subject.
- Exclude all prior consumed/development subjects and all 100 Wave 1 subjects.
- Membership may use axis target, Rodden-AA/birth-year eligibility, deterministic hash, and the predeclared female guardrail only.
- Do **not** use event outcomes, event chronology, event pairability, astrology features, astrology scores, or sealed holdouts.
- Once this notebook freezes Wave 2 membership, do not replace Wave 2 subjects after event collection begins.

In [1]:
from pathlib import Path
from datetime import datetime
import hashlib
import json
import re
import unicodedata
import urllib.request

import numpy as np
import pandas as pd

NOTEBOOK_VERSION = "SAJU_ML_V4_UNIFIED_DEV_WAVE2_SUBJECT_ROSTER_FREEZE_20260816"

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p] + list(p.parents):
        if (candidate / "saju_engine.py").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the Chartpalja saju repo.")

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def norm_name(x):
    s = unicodedata.normalize("NFKD", str(x))
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.casefold()
    s = re.sub(r"[^a-z0-9]+", "", s)
    return s

ROOT = find_repo_root()

WAVE1_ROSTER_DIR = ROOT / "research/ml/artifacts/v4_unified_dev_roster"
WAVE1_QA_DIR = ROOT / "research/ml/artifacts/v4_unified_dev_wave1"
WAVE2_CORPUS_DIR = ROOT / "research/ml_corpus/v4_unified_dev_wave2"
OUT = ROOT / "research/ml/artifacts/v4_unified_dev_wave2_roster"
OUT.mkdir(parents=True, exist_ok=True)

SEED_PATH = WAVE2_CORPUS_DIR / "V4_UNIFIED_DEV_WAVE2_PREDECLARED_SEED_UNIVERSE_R3.csv"
CONTRACT_PATH = WAVE2_CORPUS_DIR / "V4_UNIFIED_DEV_WAVE2_COLLECTION_CONTRACT.json"
WAVE1_ROSTER_PATH = WAVE1_ROSTER_DIR / "V4_UNIFIED_DEV_SUBJECT_ROSTER_100.csv"
WAVE1_ROSTER_FREEZE_PATH = WAVE1_ROSTER_DIR / "V4_UNIFIED_DEV_SUBJECT_ROSTER_FREEZE.json"
WAVE1_DECISION_PATH = WAVE1_QA_DIR / "V4_UNIFIED_DEV_WAVE1_FREEZE_DECISION.json"

for p in [SEED_PATH, CONTRACT_PATH, WAVE1_ROSTER_PATH, WAVE1_ROSTER_FREEZE_PATH, WAVE1_DECISION_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)

seed_df = pd.read_csv(SEED_PATH)
with open(CONTRACT_PATH, encoding="utf-8") as f:
    contract = json.load(f)
with open(WAVE1_ROSTER_FREEZE_PATH, encoding="utf-8") as f:
    wave1_roster_freeze = json.load(f)
with open(WAVE1_DECISION_PATH, encoding="utf-8") as f:
    wave1_decision = json.load(f)
wave1_roster = pd.read_csv(WAVE1_ROSTER_PATH)

assert contract["status"] == "WAVE2_SUBJECT_UNIVERSE_PREDECLARED_BEFORE_EVENT_COLLECTION"
assert wave1_decision["status"] == contract["prerequisite_wave1_status"]
assert len(wave1_roster) == 100
assert wave1_roster["name"].map(norm_name).nunique() == 100
assert wave1_roster["astrology_scored"].fillna(False).astype(bool).sum() == 0

actual_wave1_hash = sha256_file(WAVE1_ROSTER_PATH)
assert actual_wave1_hash == wave1_roster_freeze["roster_sha256"], (
    "Wave 1 roster file no longer matches its freeze manifest."
)
assert actual_wave1_hash == contract["expected_wave1_roster_sha256"], (
    "Wave 1 roster hash differs from the roster used for Wave 1 event collection."
)

TARGET = {k:int(v) for k,v in contract["wave2_roster_target"].items()}
assert TARGET == {"COMPETITIVE":3, "PROJECT":8, "STATUS":33}
assert sum(TARGET.values()) == 44

assert wave1_decision["recommended_wave2_roster_axis_counts"] == TARGET
assert wave1_decision["rules"]["wave2_selection_may_use_axis_deficit"] is True
assert wave1_decision["rules"]["wave2_selection_may_use_chronology_direction"] is False
assert wave1_decision["rules"]["astrology_feature_generation_allowed"] is False

SELECTION_SEED = int(contract["selection_seed"])
BIRTH_YEAR_MIN = int(contract["birth_year_min"])
BIRTH_YEAR_MAX = int(contract["birth_year_max"])
BIRTH_URL = contract["birth_source_url"]

print("repo:", ROOT)
print("output:", OUT)
print("Wave 1 roster SHA:", actual_wave1_hash)
print("Wave 2 target:", TARGET)
print("Selection seed:", SELECTION_SEED)

repo: /Users/sangjinlee/Desktop/projects/saju
output: /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v4_unified_dev_wave2_roster
Wave 1 roster SHA: 8fc11f61c5102f3c2caa9890ec82819e164178ac0a96f4d721f4e13767fdc9a7
Wave 2 target: {'COMPETITIVE': 3, 'PROJECT': 8, 'STATUS': 33}
Selection seed: 2026081602


## 1. Exclusion universe: consumed development + all Wave 1 subjects

In [2]:
def extract_subject_names_from_corpus(path):
    if not path.exists():
        return []
    with open(path, encoding="utf-8") as f:
        obj = json.load(f)
    return [
        s.get("name")
        for s in obj.get("subjects", [])
        if s.get("name")
    ]

# Only consumed/development corpora are loaded. Sealed holdouts are never referenced.
prior_paths = {
    "V1": ROOT / "research/ml_corpus/v1/SAJU_ML_CORPUS_V1.json",
    "V2_NEW_DEV": ROOT / "research/ml_corpus/v2/SAJU_ML_CORPUS_V2_NEW_DEV.json",
    "NEW_DEV_2_CONSUMED": ROOT / "research/ml_corpus/new_dev_2/SAJU_ML_NEW_DEV_2_CORPUS.json",
    "V4_TARGET_EXPANSION": ROOT / "research/ml_corpus/v4_target_expansion/V4_TARGET_EXPANSION_CORPUS.json",
}

prior_names = []
prior_loaded_counts = {}
for label, path in prior_paths.items():
    names = extract_subject_names_from_corpus(path)
    prior_loaded_counts[label] = len(names)
    prior_names.extend(names)
    print(label, len(names), "subjects loaded for exclusion")

prior_norm = {norm_name(x) for x in prior_names}
wave1_norm = set(wave1_roster["name"].map(norm_name))

assert len(wave1_norm) == 100
assert not any("NEW_CONFIRM" in str(p).upper() for p in prior_paths.values())
assert not any("VALIDATION_B" in str(p).upper() for p in prior_paths.values())
assert not any("PUBLIC_CHECK" in str(p).upper() for p in prior_paths.values())
assert not any("PUBLIC_FINAL" in str(p).upper() for p in prior_paths.values())

print("prior unique normalized names:", len(prior_norm))
print("Wave 1 frozen names:", len(wave1_norm))
print("sealed-set paths: NOT DEFINED / NOT LOADED")

V1 94 subjects loaded for exclusion
V2_NEW_DEV 70 subjects loaded for exclusion
NEW_DEV_2_CONSUMED 100 subjects loaded for exclusion
V4_TARGET_EXPANSION 169 subjects loaded for exclusion
prior unique normalized names: 232
Wave 1 frozen names: 100
sealed-set paths: NOT DEFINED / NOT LOADED


## 2. Load the same Rodden-AA birth source used for Wave 1

In [3]:
# Prefer the exact cached Wave 1 birth dataset. This preserves the birth-source snapshot
# used when Wave 1 membership was frozen.
wave1_birth_cache = WAVE1_ROSTER_DIR / "PersonList-15k.csv"
wave2_birth_cache = OUT / "PersonList-15k.csv"

if wave1_birth_cache.exists():
    birth_path = wave1_birth_cache
else:
    birth_path = wave2_birth_cache
    if not birth_path.exists():
        print("Wave 1 cache missing; downloading the predeclared public birth source...")
        urllib.request.urlretrieve(BIRTH_URL, birth_path)

actual_birth_hash = sha256_file(birth_path)
expected_birth_hash = contract.get("expected_birth_dataset_sha256")
if expected_birth_hash:
    assert actual_birth_hash == expected_birth_hash, (
        "Birth dataset hash changed. Do not continue with a different source snapshot. "
        "Use the same PersonList-15k.csv snapshot used to create this Wave 2 package."
    )

birth = pd.read_csv(birth_path)
required_cols = {"RowKey","BirthTime","Gender","Name","Notes"}
assert required_cols.issubset(birth.columns)

birth = birth[birth["Notes"].astype(str).str.contains("AA", regex=False)].copy()
birth["norm_name"] = birth["Name"].map(norm_name)

print("birth source:", birth_path)
print("birth SHA:", actual_birth_hash)
print("Rodden-AA rows:", len(birth))

birth source: /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v4_unified_dev_roster/PersonList-15k.csv
birth SHA: ca28a3fea1250b2ea01eb06a54b4921ec0bf790fdc14c5c561006f1c592c634b
Rodden-AA rows: 15790


## 3. Resolve predeclared candidates and enforce fresh-subject eligibility

In [4]:
def parse_birth_blob(blob):
    obj = json.loads(str(blob))
    std = obj["StdTime"]
    loc = obj["Location"]
    m = re.match(
        r"^(\d{1,2}):(\d{2})\s+(\d{2})/(\d{2})/(\d{4})\s+([+-]\d{2}:\d{2})$",
        std.strip()
    )
    if not m:
        raise ValueError("Unexpected StdTime: %r" % std)

    hh, mm, dd, mo, yyyy, offset = m.groups()
    return {
        "birth_date": "%04d-%02d-%02d" % (int(yyyy), int(mo), int(dd)),
        "birth_time": "%02d:%02d" % (int(hh), int(mm)),
        "utc_offset": offset,
        "birth_year": int(yyyy),
        "birth_place": loc["Name"],
        "longitude": float(loc["Longitude"]),
        "latitude": float(loc["Latitude"]),
    }

parsed_rows = []
for _, r in birth.iterrows():
    try:
        p = parse_birth_blob(r["BirthTime"])
    except Exception:
        continue
    parsed_rows.append({
        "source_row_key": r["RowKey"],
        "source_name": r["Name"],
        "gender": str(r["Gender"]).lower(),
        "norm_name": r["norm_name"],
        **p,
    })

birth_parsed = pd.DataFrame(parsed_rows)

# Avoid silently resolving ambiguous identical normalized names.
birth_name_counts = birth_parsed.groupby("norm_name").size()
ambiguous_birth_names = set(birth_name_counts[birth_name_counts > 1].index)
birth_unique = birth_parsed[~birth_parsed["norm_name"].isin(ambiguous_birth_names)].copy()

required_seed_cols = {
    "axis","seed_rank","name","seed_origin","selection_information_used"
}
assert required_seed_cols.issubset(seed_df.columns)
assert seed_df["axis"].isin(TARGET).all()

# Ensure forbidden outcome/chronology columns were not smuggled into the roster seed file.
forbidden_tokens = [
    "positive_year","negative_year","event_year","positive_earlier","chronology",
    "pairable","astrology_score","astrology_feature"
]
lower_cols = [str(c).lower() for c in seed_df.columns]
assert not any(any(tok in c for tok in forbidden_tokens) for c in lower_cols)

seed_df = seed_df.copy()
seed_df["norm_name"] = seed_df["name"].map(norm_name)
assert seed_df.groupby("norm_name")["axis"].nunique().max() == 1, "A candidate appears under multiple axes."

resolved = seed_df.merge(
    birth_unique,
    on="norm_name",
    how="left",
    suffixes=("_seed","_birth"),
)

resolved["prior_used"] = resolved["norm_name"].isin(prior_norm)
resolved["wave1_used"] = resolved["norm_name"].isin(wave1_norm)
resolved["birth_in_range"] = resolved["birth_year"].between(
    BIRTH_YEAR_MIN, BIRTH_YEAR_MAX, inclusive="both"
)
resolved["ambiguous_birth_name"] = resolved["norm_name"].isin(ambiguous_birth_names)

resolved["eligible"] = (
    resolved["source_row_key"].notna()
    & ~resolved["prior_used"]
    & ~resolved["wave1_used"]
    & resolved["birth_in_range"]
    & ~resolved["ambiguous_birth_name"]
)

resolution_summary = (
    resolved.groupby("axis")
    .agg(
        seeds=("name","size"),
        resolved_unique_aa=("source_row_key", lambda x: x.notna().sum()),
        prior_used=("prior_used","sum"),
        wave1_used=("wave1_used","sum"),
        ambiguous_birth_names=("ambiguous_birth_name","sum"),
        eligible=("eligible","sum"),
    )
    .reset_index()
)

display(resolution_summary)

for axis, n in TARGET.items():
    available = int(resolution_summary.loc[resolution_summary.axis==axis, "eligible"].iloc[0])
    if available < n:
        raise RuntimeError(
            "%s has only %d fresh eligible seed subjects; need %d. "
            "Do NOT lower the target or use event outcomes. Expand the role-only seed universe "
            "before Wave 2 event collection." % (axis, available, n)
        )

resolved.to_csv(OUT / "V4_UNIFIED_DEV_WAVE2_SEED_RESOLUTION_FULL.csv", index=False)
resolution_summary.to_csv(OUT / "V4_UNIFIED_DEV_WAVE2_SEED_RESOLUTION_AUDIT.csv", index=False)

,axis,seeds,resolved_unique_aa,prior_used,wave1_used,ambiguous_birth_names,eligible
0,COMPETITIVE,304,59,8,3,0,48
1,PROJECT,193,76,15,1,0,58
2,STATUS,418,69,7,19,0,47


## 4. Deterministic Wave 2 selection with predeclared gender guardrail

In [5]:
def deterministic_key(name, axis):
    raw = ("%s|WAVE2|%s|%s" % (SELECTION_SEED, axis, norm_name(name))).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

eligible = resolved[resolved["eligible"]].copy()
eligible["det_key"] = [
    deterministic_key(n, a)
    for n, a in zip(eligible["name"], eligible["axis"])
]

selected_parts = []
for axis, target_n in TARGET.items():
    pool = eligible[eligible["axis"] == axis].copy().sort_values("det_key")

    # Predeclared representation guardrail; never based on event outcomes.
    female_target = int(np.ceil(target_n * float(contract["selection_rules"]["female_guardrail_share"])))
    women = pool[pool["gender"].str.startswith("female")].head(female_target)
    chosen = women.copy()

    remaining = pool[~pool["norm_name"].isin(chosen["norm_name"])].copy()
    chosen = pd.concat([chosen, remaining.head(target_n - len(chosen))])

    if len(chosen) != target_n:
        raise RuntimeError("%s selection failed: %d / %d" % (axis, len(chosen), target_n))

    selected_parts.append(chosen)

roster = pd.concat(selected_parts, ignore_index=True)

assert len(roster) == 44
assert roster["norm_name"].nunique() == 44
assert not roster["prior_used"].any()
assert not roster["wave1_used"].any()
assert not roster["norm_name"].isin(wave1_norm).any()

roster["subject_id"] = [
    "V4UDW2_%03d" % (i+1)
    for i in range(len(roster))
]
roster["preassigned_axis"] = roster["axis"]
roster["rodden_rating"] = "AA"
roster["birth_source"] = "VedAstro 15K / Rodden AA"
roster["birth_source_url"] = BIRTH_URL
roster["event_collection_started"] = False
roster["astrology_scored"] = False
roster["wave"] = 2

cols = [
    "subject_id","name","source_name","gender","preassigned_axis",
    "birth_date","birth_time","utc_offset","birth_place","latitude","longitude",
    "rodden_rating","source_row_key","birth_source","birth_source_url",
    "seed_origin","seed_rank","event_collection_started","astrology_scored",
    "wave","det_key",
]
roster = roster[cols].sort_values(["preassigned_axis","det_key"]).reset_index(drop=True)

axis_counts = roster["preassigned_axis"].value_counts().to_dict()
assert axis_counts == TARGET

summary = (
    roster.groupby("preassigned_axis")
    .agg(
        n_subjects=("subject_id","size"),
        n_female=("gender", lambda x: x.str.startswith("female").sum()),
        birth_year_min=("birth_date", lambda x: pd.to_datetime(x).dt.year.min()),
        birth_year_max=("birth_date", lambda x: pd.to_datetime(x).dt.year.max()),
    )
    .reset_index()
)
summary["female_share"] = summary["n_female"] / summary["n_subjects"]

display(summary)
display(roster[["subject_id","name","preassigned_axis","gender","birth_date","seed_origin"]])

,preassigned_axis,n_subjects,n_female,birth_year_min,birth_year_max,female_share
0,COMPETITIVE,3,1,1940,1969,0.333333
1,PROJECT,8,4,1907,1954,0.500000
2,STATUS,33,2,1904,1970,0.060606


,subject_id,name,preassigned_axis,gender,birth_date,seed_origin
0,V4UDW2_002,Fran Tarkenton,COMPETITIVE,male,1940-02-03,WAVE2_ROLE_ONLY_EXPANSION
1,V4UDW2_003,Wade Boggs,COMPETITIVE,male,1958-06-15,WAVE2_ROLE_ONLY_EXPANSION
2,V4UDW2_001,Nancy Kerrigan,COMPETITIVE,female,1969-10-13,WAVE2_ROLE_ONLY_EXPANSION
3,V4UDW2_006,Dennis Hopper,PROJECT,male,1936-05-17,WAVE2_ROLE_ONLY_EXPANSION
4,V4UDW2_007,Meat Loaf,PROJECT,male,1947-09-27,WAVE2_ROLE_ONLY_EXPANSION
5,V4UDW2_004,Linda Ronstadt,PROJECT,female,1946-07-15,WAVE2_ROLE_ONLY_EXPANSION
6,V4UDW2_005,Annie Lennox,PROJECT,female,1954-12-25,WAVE2_ROLE_ONLY_EXPANSION
7,V4UDW2_008,Jessica Lange,PROJECT,female,1949-04-20,WAVE2_ROLE_ONLY_EXPANSION
8,V4UDW2_009,Jerry Lee Lewis,PROJECT,male,1935-09-29,WAVE2_ROLE_ONLY_EXPANSION
9,V4UDW2_010,Laurence Olivier,PROJECT,male,1907-05-22,WAVE2_ROLE_ONLY_EXPANSION


## 5. Freeze manifest and event-intake template

In [6]:
roster_path = OUT / "V4_UNIFIED_DEV_WAVE2_SUBJECT_ROSTER_44.csv"
roster.to_csv(roster_path, index=False)

roster_hash = sha256_file(roster_path)
seed_hash = sha256_file(SEED_PATH)
contract_hash = sha256_file(CONTRACT_PATH)
wave1_decision_hash = sha256_file(WAVE1_DECISION_PATH)

freeze = {
    "version": "V4_UNIFIED_DEV_WAVE2_SUBJECT_ROSTER_FREEZE_V1",
    "notebook_version": NOTEBOOK_VERSION,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "status": "V4_UNIFIED_DEV_WAVE2_SUBJECT_UNIVERSE_FROZEN",
    "wave": 2,
    "n_subjects": int(len(roster)),
    "axis_counts": axis_counts,
    "roster_sha256": roster_hash,
    "seed_universe_sha256": seed_hash,
    "collection_contract_sha256": contract_hash,
    "wave1_roster_sha256": actual_wave1_hash,
    "wave1_freeze_decision_sha256": wave1_decision_hash,
    "selection_seed": SELECTION_SEED,
    "birth_dataset_sha256": actual_birth_hash,
    "birth_source_url": BIRTH_URL,
    "prior_development_unique_names_excluded": int(len(prior_norm)),
    "wave1_subjects_excluded": int(len(wave1_norm)),
    "event_collection_started": False,
    "astrology_scored": False,
    "membership_conditioned_on_chronology": False,
    "membership_conditioned_on_event_pairability": False,
    "membership_conditioned_on_astrology": False,
    "sealed_holdouts_loaded": False,
    "holdout_integrity": {
        "NEW_CONFIRM_loaded": False,
        "Validation_B_loaded": False,
        "Public_CHECK_loaded": False,
        "Public_FINAL_loaded": False,
    },
    "next_rule": (
        "Do not change Wave 2 membership. Perform a bounded major-event source sweep on these "
        "44 frozen subjects only, within each preassigned axis. Keep both pairable and unpairable "
        "subjects. Do not generate astrology features. After Wave 2 event freeze/QA, combine Wave 1 "
        "and Wave 2 usable pairs; only if all target axis counts are met may the project proceed to "
        "a separate combined-corpus feature-generation gate."
    ),
}

with open(OUT / "V4_UNIFIED_DEV_WAVE2_SUBJECT_ROSTER_FREEZE.json","w",encoding="utf-8") as f:
    json.dump(freeze, f, ensure_ascii=False, indent=2)

# 3 positive + 3 negative candidate slots per frozen subject.
# Additional verified candidates may be appended; inconvenient events must not be deleted.
intake_rows = []
for _, s in roster.iterrows():
    for polarity in ["positive","negative"]:
        for slot in range(1,4):
            intake_rows.append({
                "subject_id": s["subject_id"],
                "name": s["name"],
                "preassigned_axis": s["preassigned_axis"],
                "candidate_polarity": polarity,
                "slot": slot,
                "event_year": "",
                "event_date_if_known": "",
                "event_type": "",
                "event_description": "",
                "primary_source_url": "",
                "secondary_source_url": "",
                "source_quality": "",
                "taxonomy_confirmed": False,
                "polarity_confirmed": False,
                "source_confirmed": False,
                "exclude_reason_if_invalid": "",
                "notes": "",
            })

intake = pd.DataFrame(intake_rows)
intake.to_csv(OUT / "V4_UNIFIED_DEV_WAVE2_EVENT_CANDIDATE_INTAKE.csv", index=False)

summary.to_csv(OUT / "V4_UNIFIED_DEV_WAVE2_ROSTER_AXIS_AUDIT.csv", index=False)

print(json.dumps(freeze, ensure_ascii=False, indent=2))
print("event intake rows:", len(intake))

{
  "version": "V4_UNIFIED_DEV_WAVE2_SUBJECT_ROSTER_FREEZE_V1",
  "notebook_version": "SAJU_ML_V4_UNIFIED_DEV_WAVE2_SUBJECT_ROSTER_FREEZE_20260816",
  "created_at": "2026-08-16T21:07:35",
  "status": "V4_UNIFIED_DEV_WAVE2_SUBJECT_UNIVERSE_FROZEN",
  "wave": 2,
  "n_subjects": 44,
  "axis_counts": {
    "STATUS": 33,
    "PROJECT": 8,
    "COMPETITIVE": 3
  },
  "roster_sha256": "afd9cf204f26ccf00b384e41d8de06d708f641892211f5226462b178caaf1e27",
  "seed_universe_sha256": "ee3eaf2c907462a4152140350219787630dec7c82c00dd0a0a32f378c4666f29",
  "collection_contract_sha256": "1b25f8ab9321c18b72a28e0fd30439c343475ba87345a199e83f0497bb0a638e",
  "wave1_roster_sha256": "8fc11f61c5102f3c2caa9890ec82819e164178ac0a96f4d721f4e13767fdc9a7",
  "wave1_freeze_decision_sha256": "599e16a250c2c301901334afa0112ab7af084b576e725a81344ab863ce8920e7",
  "selection_seed": 2026081602,
  "birth_dataset_sha256": "ca28a3fea1250b2ea01eb06a54b4921ec0bf790fdc14c5c561006f1c592c634b",
  "birth_source_url": "https://huggi

## 6. Final hard-stop QA

In [7]:
# Membership integrity
assert len(roster) == 44
assert roster["subject_id"].nunique() == 44
assert roster["name"].map(norm_name).nunique() == 44
assert roster["preassigned_axis"].value_counts().to_dict() == TARGET

# Freshness
assert not roster["name"].map(norm_name).isin(wave1_norm).any()
assert not roster["name"].map(norm_name).isin(prior_norm).any()

# Birth eligibility
birth_years = pd.to_datetime(roster["birth_date"]).dt.year
assert birth_years.between(BIRTH_YEAR_MIN, BIRTH_YEAR_MAX).all()
assert (roster["rodden_rating"] == "AA").all()

# No outcome / chronology leakage in roster
forbidden_roster_cols = [
    "positive_year","negative_year","event_year","positive_earlier","chronology",
    "pairable","astrology_score","astrology_feature"
]
assert not any(c in roster.columns for c in forbidden_roster_cols)

# No scoring or event collection before freeze
assert not roster["event_collection_started"].astype(bool).any()
assert not roster["astrology_scored"].astype(bool).any()

# Intake is blank and mechanically generated after membership freeze
assert len(intake) == 44 * 2 * 3
assert (intake["event_year"].astype(str).str.len() == 0).all()
assert (intake["event_description"].astype(str).str.len() == 0).all()

print("STATUS: V4_UNIFIED_DEV_WAVE2_SUBJECT_UNIVERSE_FROZEN")
print("Wave 2 membership is now immutable once event collection begins.")
print("Do NOT generate astrology features yet.")
print()
print("Send back:")
print("- V4_UNIFIED_DEV_WAVE2_SUBJECT_ROSTER_44.csv")
print("- V4_UNIFIED_DEV_WAVE2_SUBJECT_ROSTER_FREEZE.json")
print("- V4_UNIFIED_DEV_WAVE2_SEED_RESOLUTION_AUDIT.csv")

STATUS: V4_UNIFIED_DEV_WAVE2_SUBJECT_UNIVERSE_FROZEN
Wave 2 membership is now immutable once event collection begins.
Do NOT generate astrology features yet.

Send back:
- V4_UNIFIED_DEV_WAVE2_SUBJECT_ROSTER_44.csv
- V4_UNIFIED_DEV_WAVE2_SUBJECT_ROSTER_FREEZE.json
- V4_UNIFIED_DEV_WAVE2_SEED_RESOLUTION_AUDIT.csv
